# Betting Backtest

This notebook covers odds ingestion, source selection, implied-probability cleaning, merge diagnostics, the first exploratory Random Forest betting backtest, and post-hoc diagnostics on why that backtest lost money. It does **not** retrain or tune the model.

The betting layer inherits the audited modeling cutoff as well: only fights on or after `2010-01-01` enter the predictive workflow used here. All model probabilities used here come from the post-2009 training and test pipeline; this notebook does not re-open the pre-2010 regime that was excluded for target-consistency reasons.




In [ ]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.train_baseline_models import (
    run_odds_ingestion_pipeline,
    run_betting_backtest,
    run_betting_failure_diagnostics,
)


## Odds Data Source

We first locate the odds file, standardize its schema, and summarize available sources. The current pipeline keeps the matching logic auditable and does not average books in the first pass.


In [ ]:
odds_results = run_odds_ingestion_pipeline()
odds_results


## Cleaning Odds

The cleaned selected-source file standardizes event date, fighter names, source, and prices. Fighter keys are normalized with uppercasing, punctuation removal, whitespace cleanup, and accent normalization.


In [ ]:
pd.read_csv(repo_root / 'data' / 'ufc_odds_clean_selected_source.csv').head()


## Choosing a Bookmaker/Source

For the first pass we pick one source instead of averaging across books. That keeps the prices closer to something tradable and easier to audit. If the selected source is `zewnetrzne`, we treat it as an external or aggregated feed.


In [ ]:
pd.read_csv(repo_root / 'outputs' / 'odds_source_coverage.csv').head(20)


In [ ]:
(repo_root / 'outputs' / 'selected_odds_source.txt').read_text()


## Implied Probability and Vig Removal

The cleaned odds are converted into raw implied probabilities and then normalized to no-vig probabilities. For decimal odds, implied probability is `1 / decimal_odds`. The no-vig probabilities divide each side by the overround.


In [ ]:
pd.read_csv(repo_root / 'data' / 'ufc_odds_clean_with_implied_probs.csv')[['event_date', 'fighter_1', 'fighter_2', 'fighter_1_odds', 'fighter_2_odds', 'fighter_1_implied_prob_novig', 'fighter_2_implied_prob_novig', 'overround']].head()


## Merge Diagnostics

The odds are then oriented to red/blue corners and merged into the modeling dataset using event date plus normalized unordered fighter-pair keys. This is still exact matching, so it can miss fights with naming mismatches.

Because the modeling universe now starts in 2010, unmatched early-2010 fights mostly reflect odds coverage gaps rather than missing pre-2010 model rows.



In [ ]:
print((repo_root / 'outputs' / 'odds_merge_diagnostics.txt').read_text())


## Edge Explanation

The backtest uses the frozen Random Forest probability for the red fighter and compares it to the no-vig market implied probability.

- `model_prob_red = RF predicted probability`
- `model_prob_blue = 1 - model_prob_red`
- `edge_red = model_prob_red - red_implied_prob_novig`
- `edge_blue = model_prob_blue - blue_implied_prob_novig`

For each fight, the strategy can bet at most one side: the side with the larger positive edge above the threshold.


## Flat Betting Results

Flat staking uses a constant $1 stake per bet. Profit is `odds - 1` on a win and `-1` on a loss.


In [ ]:
backtest_results = run_betting_backtest()
backtest_results['flat_results']


## Kelly Betting Results

The exploratory Kelly pass uses capped full Kelly fractions. We compute full Kelly with `f* = (b*p - q) / b`, cap the full Kelly at 0.25, and then apply fractional Kelly multipliers of 0.25 and 0.50. Negative Kelly values are clipped to zero.


In [ ]:
backtest_results['kelly_results']


## Why Did the Betting Backtest Fail?

This section is diagnostic only. We are not changing the model, tuning thresholds, or feeding these results back into training. We use the best flat-betting threshold from the exploratory pass as a single concrete policy to inspect.


In [ ]:
diagnostics = run_betting_failure_diagnostics()
diagnostics['best_threshold']


### Segment Tables


In [ ]:
diagnostics['segment_tables']['side']


In [ ]:
diagnostics['segment_tables']['odds_bucket']


In [ ]:
diagnostics['segment_tables']['market_prob_bucket']


In [ ]:
diagnostics['segment_tables']['model_prob_bucket']


In [ ]:
diagnostics['segment_tables']['edge_bucket']


### Edge Quality

This table checks whether larger model edge actually led to better realized outcomes and better profits. If edge quality is weak, the model-vs-market disagreement may not be actionable.


In [ ]:
diagnostics['edge_quality']


In [ ]:
diagnostics['edge_correlations']


### Calibration Comparison

We compare model calibration and market calibration on the subset of hold-out fights that have matched odds. This helps show whether the market or the model tracked realized outcomes more closely.


In [ ]:
diagnostics['model_calibration']


In [ ]:
diagnostics['market_calibration']


### Favorite / Longshot Analysis

This helps check whether the losses were concentrated in underdogs and especially longshots, where volatility is high and calibration errors get expensive.


In [ ]:
diagnostics['favorite_longshot']


### Time / Regime Performance


In [ ]:
diagnostics['cumulative_profit'].head()


In [ ]:
diagnostics['time_segment_performance'].head(20)


### Interpretation

The main goal here is to understand failure modes, not to rescue the strategy by tuning on the hold-out set. Any future calibration, source selection, or line-shopping work should happen on validation only before a new untouched test is used.


In [ ]:
print((repo_root / 'outputs' / 'betting_failure_diagnostics_summary.txt').read_text())


## Final Comparison Visuals

These charts summarize final model quality and the betting diagnostics using already-saved outputs only. No model retraining, retuning, or new backtest rules are introduced here.


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
figures_dir = repo_root / 'outputs' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

final_model_comparison = pd.read_csv(repo_root / 'outputs' / 'final_model_test_comparison.csv')
flat_results = pd.read_csv(repo_root / 'outputs' / 'betting_flat_backtest_results.csv')
flat_bets = pd.read_csv(repo_root / 'outputs' / 'betting_flat_backtest_bets.csv', parse_dates=['event_date'])
odds_bucket = pd.read_csv(repo_root / 'outputs' / 'betting_segment_by_odds_bucket.csv')
model_calibration = pd.read_csv(repo_root / 'outputs' / 'betting_model_calibration_on_matched_odds.csv')
market_calibration = pd.read_csv(repo_root / 'outputs' / 'betting_market_calibration_on_matched_odds.csv')
cumulative_profit = pd.read_csv(repo_root / 'outputs' / 'betting_cumulative_profit.csv', parse_dates=['event_date'])


### Final Model Comparison

These bar charts compare the three frozen model families on the final hold-out test set. Lower is better for log loss and Brier score; higher is better for ROC AUC.


In [ ]:
comparison = final_model_comparison.set_index('model_name').loc[['logistic_regression', 'random_forest', 'xgboost']]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ['tab:blue', 'tab:green', 'tab:orange']
for ax, metric, title in zip(axes, ['log_loss', 'roc_auc', 'brier_score'], ['Log Loss', 'ROC AUC', 'Brier Score']):
    ax.bar(comparison.index, comparison[metric], color=colors)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(figures_dir / '06_final_model_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()


### Cumulative Flat-Betting Profit Over Time

This shows the path of the selected flat-betting policy over the matched hold-out fights. The goal is to visualize drawdown and instability, not to optimize a new rule.


In [ ]:
best_threshold = flat_results.sort_values(['roi', 'total_profit'], ascending=[False, False]).iloc[0]['edge_threshold']
selected_flat = flat_bets[(flat_bets['edge_threshold'].round(10) == round(float(best_threshold), 10)) & (flat_bets['selected_side'] != 'skip')].copy()
selected_flat = selected_flat.sort_values('event_date')
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(selected_flat['event_date'], selected_flat['cumulative_profit'], linewidth=2)
ax.axhline(0, color='black', linewidth=1)
ax.set_title(f'Cumulative Flat-Betting Profit (threshold={best_threshold:.2f})')
ax.set_xlabel('Event date')
ax.set_ylabel('Cumulative profit ($)')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(figures_dir / '06_cumulative_flat_profit.png', dpi=150, bbox_inches='tight')
plt.show()


### ROI By Edge Threshold

This chart compares the flat-betting thresholds already evaluated in the saved backtest. It is descriptive only because these are hold-out test results.


In [ ]:
threshold_plot = flat_results.sort_values('edge_threshold')
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(threshold_plot['edge_threshold'].astype(str), threshold_plot['roi'], color='tab:purple')
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Flat-Betting ROI By Edge Threshold')
ax.set_xlabel('Edge threshold')
ax.set_ylabel('ROI')
fig.tight_layout()
fig.savefig(figures_dir / '06_roi_by_edge_threshold.png', dpi=150, bbox_inches='tight')
plt.show()


### ROI By Odds Bucket

This breakdown helps show where betting performance was strongest and weakest across price regions.


In [ ]:
bucket_plot = odds_bucket.copy()
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(bucket_plot['odds_bucket'].astype(str), bucket_plot['roi'], color='tab:brown')
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Flat-Betting ROI By Odds Bucket')
ax.set_xlabel('Odds bucket')
ax.set_ylabel('ROI')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(figures_dir / '06_roi_by_odds_bucket.png', dpi=150, bbox_inches='tight')
plt.show()


### Model vs Market Calibration On Odds-Matched Fights

These lines compare realized win rate to the model and the no-vig market probabilities on the subset of hold-out fights with matched odds.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
ax.plot(model_calibration['average_model_probability'], model_calibration['actual_win_rate'], marker='o', linewidth=2, label='Model')
ax.plot(market_calibration['average_market_probability'], market_calibration['actual_win_rate'], marker='o', linewidth=2, label='Market')
ax.set_title('Calibration on Odds-Matched Fights')
ax.set_xlabel('Average predicted probability')
ax.set_ylabel('Actual win rate')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(figures_dir / '06_model_vs_market_calibration.png', dpi=150, bbox_inches='tight')
plt.show()


## Walk-Forward Market Context

The backtest results above are based on the frozen hold-out test predictions. Separately, the repo now includes a pre-test walk-forward study that compares raw model probabilities, calibrated probabilities, and market no-vig probabilities on annual out-of-sample folds. That new layer does not rescue the backtest directly, but it does help explain why the strategy struggled.


In [ ]:
walk_forward_comparison = pd.read_csv(outputs_dir / "walk_forward_model_comparison.csv")
walk_forward_comparison


The main message is that the market remained the best standalone probability benchmark on the pre-test odds-matched sample, while the raw Random Forest lagged and only partially recovered through calibration and market blending. That lines up with the negative betting results: the model may contain some signal, but not enough independent probability edge to overcome market pricing and vig.
